# Camera sheets and the narrative in metres, with chain of thought: a pilot (run 2)

Run 2 of the camera pilot (`README.md` in this folder, `experiments/2026-11-20-camera/`). Run 1 (`explore_camera.ipynb`)
wrote every pipeline sheet in VGGT's relative units. This run writes them **in metres**: the objects, the camera
tables, the compass path and the narrative, all with the gold-free scale from typical object sizes
(`--metric-scale priors`, `mvp.benchmarks.scene_scale`), one scale per question. It also adds three
**chain-of-thought** conditions. The questions are explore200's (`explore200_ids.txt`, 25 per VSI-Bench task, from 25
scenes), as in run 1 and the structured-CoT pilot.

Every condition: Qwen3.5-2B, thinking off, 16 frames (`uniform128`), greedy decoding, batches of up to 8 questions of
one video, and one rule for reading the answer (`score_final`). The cameras are always in the objects' format.
- **Direct** (lmms-eval's prompt, up to 256 new tokens): run 1's 15 conditions, on all 200 questions:
  - `video`, copied from Drive if it checks out;
  - `obj_csv`, `obj_text`, `obj_markdown_table`: objects only;
  - `cam_csv`, `cam_markdown_table`: objects and camera tables;
  - `cam_text`: objects as sentences and the compass path;
  - `narrative`: objects and cameras in one walk;
  - the variants of the camera tables, the camera text and the narrative.
- **Chain of thought** (the eval's `--reasoning cot`, as in the structured-CoT pilot, up to 2,048 new tokens):
  - `cot_cam_csv` or `cot_cam_markdown_table`: the camera table in whichever of the two formats has the higher
    overall in this run's direct answers (`cam_csv` against `cam_markdown_table`; csv on a tie). Set `COT_TABLE` to
    choose by hand.
  - `cot_cam_text`: the compass path with the objects as sentences.
  - `cot_narrative`: the narrative.

  The CoT instructions are the structured-CoT pilot's (`cot_prompt`). The sheet is the pipeline's in metres, so they
  add the warning that it can be wrong and do not ask the model to work out a scale.

**Runs.** Each condition is timed on 5 questions of one video (a direct one after a warm-up question), and the
notebook prints the estimated time for the rest on this GPU. Above `HOURS_LIMIT` (4 h) it stops until you choose
`SUBSET`, a number of questions per question type for the CoT conditions; the direct ones always cover all 200. The
direct conditions run first, then the table format for CoT is chosen, then the three CoT conditions run.

**Summary.**
- The direct conditions as in run 1: comparisons, letter bias, numeric answers.
- Each direct condition in metres against run 1's in relative units (`RELATIVE_DIR`).
- The CoT conditions against their direct ones, with the structured-CoT pilot's reply diagnostics:
  - cut-off replies and loops;
  - the "Answer:" line;
  - whether replies use the sheet's rows, numbers and cameras.
- Bootstrap 95% intervals and example replies.

**After a crash**, run the notebook again from the top: data already in this runtime is not fetched again, a condition
whose predictions on Drive answer every question is skipped, a partial one continues from its last saved batch, the
timing is kept (`timing.json`), and so is the table format chosen for CoT (`cot_table.json`).

**Files on Google Drive**, in `MyDrive/mvp/`: `mvp-3d-vlm-camera-metres.zip`, the code, made in the repository
(committed or not) with

```
REV=$(git stash create); git archive --format=zip -o mvp-3d-vlm-camera-metres.zip ${REV:-HEAD} src experiments/2026-11-18-scene-sheet-formats requirements.txt pyproject.toml
```

the scene-sheets zip (`SHEETS_ZIP`), the video-only predictions to reuse (`ssf_explore200_results/`,
`scot_explore_results/`) and run 1's predictions (`camera_explore_results/`). VSI-Bench's `test.jsonl` and the 25 videos
come from Hugging Face. Predictions go to `MyDrive/mvp/camera_metres_explore_results/`.

## Paths

In [ ]:
USE_DRIVE = True                     # False: put the files in /content yourself (Files pane) and set DRIVE_DIR to "/content"
DRIVE_DIR = "/content/drive/MyDrive/mvp"
CODE_ZIP = f"{DRIVE_DIR}/mvp-3d-vlm-camera-metres.zip"  # the code (the first cell says how to make it)
DATA_SOURCE = "huggingface"          # "huggingface": VSI-Bench from Hugging Face + SHEETS_ZIP | "bundle": DATA_TAR from the cluster
SHEETS_ZIP = f"{DRIVE_DIR}/scene_sheets_75.zip"  # the scene-sheets zip: set to its name on your Drive
DATA_TAR = f"{DRIVE_DIR}/ssf_explore_data.tar.gz"  # only for DATA_SOURCE = "bundle"
WORK = "/content/mvp-3d-vlm"         # the code and data are unpacked here
RESULTS_DIR = f"{DRIVE_DIR}/camera_metres_explore_results" if USE_DRIVE else "/content/camera_metres_explore_results"  # this notebook's predictions (on Drive, they survive a disconnect and the runs resume)
REUSE_VIDEO = [f"{DRIVE_DIR}/ssf_explore200_results/A_video.jsonl",  # explore200's video-only predictions
               f"{DRIVE_DIR}/scot_explore_results/A_video.jsonl"]    # the structured-CoT pilot's (the first that checks out is used)
RELATIVE_DIR = f"{DRIVE_DIR}/camera_explore_results"  # run 1's predictions, in relative units, for metres − relative

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# Every condition has the same model, frames, batches and answer rule (COMMON); they differ in the scene information
# between the video and the question and, for the CoT ones, in the instructions after it and the token limit.
MODEL = "Qwen/Qwen3.5-2B"            # thinking off: the eval turns it off for Qwen3.5
SSF = f"{WORK}/experiments/2026-11-18-scene-sheet-formats"  # the format experiment: the 200 questions
IDS_PATH = f"{SSF}/explore200_ids.txt"
SHEETS_ROOT = f"{WORK}/data/scene_sheets_75"
METRIC_SCALE = "priors"              # every pipeline sheet approximately in metres (mvp.benchmarks.scene_scale, no gold)
DIRECT_TOKENS = 256                  # direct answers are a letter or a number; this only stops a reply that reasons
COT_TOKENS = 2048                    # as the structured-CoT pilot: its loops, not the limit, cut replies off
BATCH_SIZE = 8                       # the eval's default, as run 1, explore200 and the CoT pilot (the reuse check needs it)
TOKEN_BUDGET = 120000                # the eval's default --token-budget (input tokens per batch)
COMMON = ["--model", MODEL, "--nframes", "16", "--sampling", "uniform128", "--final-answer",
          "--batch-size", str(BATCH_SIZE), "--token-budget", str(TOKEN_BUDGET)]
DEFAULTS = {"cameras": False, "time_unit": "seconds", "visibility": "bridged", "step_order": "move-turn",
            "object_directions": "floor"}
DIRECT = {                           # run 1's conditions, in the order they run: the main ones first, then the variants
    "video": None,
    "obj_csv": {"fmt": "csv"},
    "cam_csv": {"fmt": "csv", "cameras": True},
    "obj_text": {"fmt": "text"},
    "cam_text": {"fmt": "text", "cameras": True},                # the cameras as text: the compass path
    "narrative": {"fmt": "narrative"},                           # objects and cameras in one walk
    "obj_markdown_table": {"fmt": "markdown_table"},
    "cam_markdown_table": {"fmt": "markdown_table", "cameras": True},
    "cam_csv_frames": {"fmt": "csv", "cameras": True, "time_unit": "frames"},
    "cam_text_turn_move": {"fmt": "text", "cameras": True, "step_order": "turn-move"},
    "cam_text_frames": {"fmt": "text", "cameras": True, "time_unit": "frames"},
    "narrative_turn_move": {"fmt": "narrative", "step_order": "turn-move"},
    "narrative_camera_dirs": {"fmt": "narrative", "object_directions": "camera"},
    "narrative_frames": {"fmt": "narrative", "time_unit": "frames"},
    "narrative_exact": {"fmt": "narrative", "visibility": "exact"},
}
COT_BASES = {"cot_cam_csv": "cam_csv", "cot_cam_markdown_table": "cam_markdown_table",  # one of these two runs
             "cot_cam_text": "cam_text", "cot_narrative": "narrative"}                  # CoT name -> its direct condition
TABLE_COT = {"csv": "cot_cam_csv", "markdown_table": "cot_cam_markdown_table"}
COT_TABLE = None                     # None: the table format whose direct cam_<format> has the higher overall here
                                     # (csv on a tie); "csv" or "markdown_table" to choose by hand
CONDITIONS = {**DIRECT, **{n: DIRECT[b] for n, b in COT_BASES.items()}}
COMPARISONS = [                      # direct, first minus second (README, Comparisons)
    ("cam_csv", "obj_csv"), ("cam_markdown_table", "obj_markdown_table"), ("cam_text", "obj_text"),  # do cameras help
    ("narrative", "cam_text"), ("narrative", "obj_text"), ("narrative", "obj_csv"),                   # the narrative
    ("cam_csv_frames", "cam_csv"), ("cam_text_turn_move", "cam_text"), ("cam_text_frames", "cam_text"),  # the options
    ("narrative_turn_move", "narrative"), ("narrative_camera_dirs", "narrative"), ("narrative_frames", "narrative"),
    ("narrative_exact", "narrative"),
]
REUSE = True                         # False: run video here instead of copying it
TIMING_N = 5                         # questions timed per condition before the full runs: one video, one batch
HOURS_LIMIT = 4                      # an estimate above this stops the notebook to let you choose SUBSET
SUBSET = None                        # for the CoT conditions (the direct ones always run on all 200): None: all 200 if
                                     # everything fits in HOURS_LIMIT, else stop and ask; "all": all 200 anyway; an
                                     # int k: k questions per question type (Runs)


def opts(name):
    """The condition's sheet options with the defaults filled in, or None for video only."""
    o = CONDITIONS[name]
    return None if o is None else {**DEFAULTS, **o}


def reasoning(name):
    return "cot" if name in COT_BASES else "direct"


def max_tokens(name):
    return COT_TOKENS if name in COT_BASES else DIRECT_TOKENS


def family(name):
    o = opts(name)
    if o is None:
        return "video only"
    kind = ("narrative" if o["fmt"] == "narrative" else "objects only" if not o["cameras"]
            else "camera text" if o["fmt"] == "text" else "camera table")
    return kind + (", CoT" if name in COT_BASES else "")


def eval_args(name):
    """The eval's arguments for a condition, without --ids and --out. Every sheet option is given explicitly."""
    o = opts(name)
    args = [*COMMON, "--max-new-tokens", str(max_tokens(name)), "--reasoning", reasoning(name)]
    if o:
        args += ["--sheet-source", "pipeline", "--sheet-format", o["fmt"], "--sheets-root", SHEETS_ROOT,
                 "--time-unit", o["time_unit"], "--visibility", o["visibility"], "--step-order", o["step_order"],
                 "--object-directions", o["object_directions"], "--metric-scale", METRIC_SCALE]
        args += ["--cameras"] if o["cameras"] else []
    return args


def cot_comparisons(cot_run):
    """The CoT comparisons, first minus second: each CoT condition against its direct one, then the three against
    each other."""
    table, text, walk = cot_run
    return ([(c, COT_BASES[c]) for c in cot_run] + [(text, table), (walk, text), (walk, table)])


assert all(x in DIRECT and y in DIRECT for x, y in COMPARISONS)

## Setup

In [ ]:
import os, shutil, subprocess, sys, zipfile

os.makedirs(WORK, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall(WORK)
    CODE_COMMIT = z.comment.decode() or "unknown"  # git archive writes the commit it archived as the zip's comment
print(f"code: {CODE_ZIP}, commit {CODE_COMMIT}")
# what this notebook needs from the code: the camera and narrative options, the cameras and the narrative in
# metres, reply lengths in the predictions, the processor's batch, the 200 questions
needs = [("src/mvp/eval/vsibench.py", "--object-directions"), ("src/mvp/eval/vsibench.py", "hit_token_limit"),
         ("src/mvp/benchmarks/scene_sheet.py", "DIRECTIONS_PRIORS"),  # the cameras and the narrative in metres
         ("src/mvp/benchmarks/scene_sheet.py", "def render_narrative"), ("src/mvp/models/qwen_vl.py", "def video_inputs"),
         ("experiments/2026-11-18-scene-sheet-formats/explore200_ids.txt", "")]
old = sorted({f for f, text in needs if not os.path.exists(f"{WORK}/{f}") or text not in open(f"{WORK}/{f}").read()})
assert not old, f"{CODE_ZIP} lacks what this notebook needs ({', '.join(old)}): make it again as the first cell says"

# requirements.txt pins torch 2.14.1+cu130; Colab's own torch is built for its driver, so it is kept unless this is False
KEEP_COLAB_TORCH = True
reqs = []
for line in open(f"{WORK}/requirements.txt").read().splitlines():
    line = line.split("#")[0].strip()
    if not line or (KEEP_COLAB_TORCH and (line.startswith("--extra-index-url") or line.split("==")[0] in ("torch", "torchvision"))):
        continue
    reqs.append(line)
# decord has no wheels for recent Python versions; eva-decord is a drop-in build imported under the same name
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[r for r in reqs if r != "decord"]], check=True)
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "decord"]).returncode != 0:
    print("decord has no wheel for this Python; installing eva-decord instead")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "eva-decord"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", WORK], check=True)
sys.path.insert(0, f"{WORK}/src")
print("installed:", ", ".join(reqs))

In [ ]:
import torch

print("torch", torch.__version__, "CUDA", torch.version.cuda)
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > GPU (L4 or A100)"
GPU = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
print(f"GPU: {GPU}, compute capability {cap[0]}.{cap[1]}, {torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB")
if "T4" in GPU or cap < (8, 0):
    print("WARNING: this GPU has no native bfloat16 (T4 or older). The model runs in bfloat16, which is emulated here "
          "and several times slower; switch to an L4 or A100 if you can.")

## Data

Laid out as the code expects it: `data/raw/VSI-Bench/` (`test.jsonl`, `<dataset>/<scene>.mp4`),
`data/vggt/<dataset>/<scene>/` (`frame_indices.npy`, `meta.json`) and `data/scene_sheets_75/` (`index.csv`, the
questions' pipeline sheet folders with `objects.csv` and `cameras.csv`, each scene's `adapter_report.json`).

In [ ]:
import csv, io, json, shutil, tarfile, zipfile
from pathlib import Path

import numpy as np

VSI_REPO = "nyu-visionx/VSI-Bench"
RAW, VGGT, SHEETS = Path(WORK) / "data/raw/VSI-Bench", Path(WORK) / "data/vggt", Path(SHEETS_ROOT)
IDS = [int(x) for x in Path(IDS_PATH).read_text().split()]


def save(data, dest):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(data)


def export_frames(video, max_frames=128):
    """frame_indices and fps exactly as the VGGT export takes them (mvp.reconstruction.extract.load_frames): decord's
    frame count and average fps, and min(128, N) frames spaced by np.linspace(0, N - 1, dtype=int)."""
    from decord import VideoReader, cpu
    vr = VideoReader(str(video), ctx=cpu(0))
    n = len(vr)
    idx = np.linspace(0, n - 1, num=min(max_frames, n), dtype=int)
    return idx, n, float(vr.get_avg_fps())


if DATA_SOURCE == "bundle":
    with tarfile.open(DATA_TAR) as t:
        t.extractall(WORK)
    print("unpacked", DATA_TAR)
else:
    from huggingface_hub import HfFileSystem, hf_hub_download

    # 1. the sheets: index.csv, every pipeline sheet folder of the questions' scenes (objects.csv, cameras.csv,
    #    question.json: the metre scale reads every pipeline sheet of a scene, scene_scale.scene_longest) and each
    #    scene's adapter_report.json. No gold sheets: no condition has one.
    with zipfile.ZipFile(SHEETS_ZIP) as z:
        names = z.namelist()
        index_name = min((n for n in names if n.endswith("index.csv")), key=len)  # the top-level one
        prefix = index_name[: -len("index.csv")]
        rows = {int(r["id"]): r for r in csv.DictReader(io.TextIOWrapper(z.open(index_name), encoding="utf-8"))}
        absent = [q for q in IDS if q not in rows]
        assert not absent, f"{SHEETS_ZIP}: questions {absent} are not in its index.csv"
        scenes = sorted({rows[q]["scene"] for q in IDS})
        wanted = {r["pipeline_sheet"] for r in rows.values() if r["scene"] in scenes}
        wanted |= {f"{s.replace('/', '__')}/subset-uniform16/scene/adapter_report.json" for s in scenes}
        save(z.read(index_name), SHEETS / "index.csv")
        n_files = 0
        for name in names:
            parts = name[len(prefix):].split("/")
            if (name.startswith(prefix) and not name.endswith("/")
                    and any("/".join(parts[:i]) in wanted for i in range(1, len(parts) + 1))):
                save(z.read(name), SHEETS.joinpath(*parts))
                n_files += 1
        print(f"sheets: index.csv and {n_files} files: every pipeline sheet of the {len(scenes)} scenes of the {len(IDS)} questions")
        # 2. frame_indices.npy and meta.json: from the zip if it has them for these scenes
        in_zip = {}
        for s in scenes:
            for f in ("frame_indices.npy", "meta.json"):
                hits = [n for n in names if n.endswith(f"/{s}/{f}") or n.endswith(f"/{s.replace('/', '__')}/{f}")]
                if hits:
                    in_zip[(s, f)] = min(hits, key=len)
        for s in scenes:
            if (s, "frame_indices.npy") in in_zip and (s, "meta.json") in in_zip:
                for f in ("frame_indices.npy", "meta.json"):
                    save(z.read(in_zip[(s, f)]), VGGT / s / f)
                print(f"{s}: frame_indices.npy and meta.json taken from the zip ({in_zip[(s, 'meta.json')]})")

    # 3. VSI-Bench's test.jsonl and the scenes' videos, read out of the per-dataset zips without downloading them whole
    RAW.mkdir(parents=True, exist_ok=True)
    if not (RAW / "test.jsonl").exists():  # after a crash in the same runtime, what is here is not fetched again
        shutil.copy(hf_hub_download(VSI_REPO, "test.jsonl", repo_type="dataset"), RAW / "test.jsonl")
    fs = HfFileSystem()
    for dataset in sorted({s.split("/")[0] for s in scenes if not (RAW / f"{s}.mp4").exists()}):
        with fs.open(f"datasets/{VSI_REPO}/{dataset}.zip", "rb", block_size=2**20) as f, zipfile.ZipFile(f) as z:
            for s in [s for s in scenes if s.startswith(dataset + "/") and not (RAW / f"{s}.mp4").exists()]:
                save(z.read(f"{s}.mp4"), RAW / f"{s}.mp4")
                print(f"{s}.mp4: {(RAW / f'{s}.mp4').stat().st_size / 2**20:.1f} MB from {dataset}.zip")

    # 4. recompute frame_indices.npy and meta.json where the zip had none
    for s in scenes:
        if (VGGT / s / "meta.json").exists() and (VGGT / s / "frame_indices.npy").exists():
            continue
        idx, n, fps = export_frames(RAW / f"{s}.mp4")
        (VGGT / s).mkdir(parents=True, exist_ok=True)
        np.save(VGGT / s / "frame_indices.npy", idx.astype(np.int32))
        (VGGT / s / "meta.json").write_text(json.dumps({
            "video": f"data/raw/VSI-Bench/{s}.mp4", "num_video_frames": n, "video_fps": fps, "num_frames": len(idx),
            "recomputed": "explore_camera_metres.ipynb: frame_indices and video_fps as mvp.reconstruction.extract.load_frames "
                          "computes them; no VGGT outputs"}, indent=1))
        print(f"{s}: recomputed from the video: {n} frames, {fps} fps, {len(idx)} frame indices")

In [ ]:
# The video's "<t seconds>" labels come from the processor, from the fps the eval's video reader reports
# (qwen_vl_utils uses torchcodec, decord or torchvision, the first one installed). torchcodec measures the fps over the
# whole stream (frames / duration) and decord reads it from the header, and the two can differ in the last bits of the
# float (30.000000000000004 against 30.0), which flips a pair's label when its mean time is on a .x5 boundary. The sheets'
# timestamps (scene_sheet.frame_timestamps) use meta.json's video_fps, so it is set here to exactly what the eval's
# reader reports, read with the eval's own QwenVL.load_video; the export's value is kept as export_video_fps.
import inspect
import json
import sys

sys.path.insert(0, f"{WORK}/src")
from mvp.benchmarks import scene_sheet as ss
from mvp.models.qwen_vl import QwenVL
from qwen_vl_utils.vision_process import get_video_reader_backend

reader = QwenVL.__new__(QwenVL)  # load_video only needs QwenVL's pixel limits, not the model or the processor
limits = inspect.signature(QwenVL.__init__).parameters
reader.min_pixels, reader.max_pixels = limits["min_pixels"].default, limits["max_pixels"].default
backend = get_video_reader_backend()
index = ss.load_index(SHEETS)
changed = []
for s in sorted({index[q]["scene"] for q in IDS}):
    path = VGGT / s / "meta.json"
    meta = json.loads(path.read_text())
    _, video_meta, _ = reader.load_video(str(RAW / f"{s}.mp4"), nframes=ss.N_FRAMES)
    fps = float(video_meta["fps"])
    meta.setdefault("export_video_fps", meta["video_fps"])
    if meta["video_fps"] != fps or meta.get("video_reader_backend") != backend:
        if meta["video_fps"] != fps:
            changed.append(f"{s}: {meta['video_fps']!r} -> {fps!r}")
        meta.update(video_fps=fps, video_reader_backend=backend)
        path.write_text(json.dumps(meta, indent=1))
ss.frame_timestamps.cache_clear()  # timestamps computed before this cell used the old fps
print(f"video reader: {backend}; meta.json's video_fps changed to the reader's in {len(changed)} scenes"
      + "".join(f"\n  {c}" for c in changed))

In [ ]:
# Check 1: the 16 frames the eval picks (uniform128: 16 of the 128 frame_indices) are the frames the sheets were made on
# (adapter_report.json's source_video_frames). If not, the sheets describe other frames than the model sees.
import sys
sys.path.insert(0, f"{WORK}/src")
from mvp.benchmarks import scene_sheet as ss
from mvp.reconstruction.sa_sampling import uniform_subset

index = ss.load_index(SHEETS)
problems = []
for s in sorted({index[q]["scene"] for q in IDS}):
    idx = np.load(VGGT / s / "frame_indices.npy")
    report = json.loads((SHEETS / s.replace("/", "__") / "subset-uniform16" / "scene" / "adapter_report.json").read_text())
    picked = idx[uniform_subset(len(idx), 16)].tolist()
    fps = json.loads((VGGT / s / "meta.json").read_text())["video_fps"]
    print(f"{s}: {len(idx)} frame indices, fps {fps}; the eval's 16 frames {picked}")
    if report["selected_frames"] != uniform_subset(128, 16):
        problems.append(f"{s}: the sheets were made on frames {report['selected_frames']} of the 128, not uniform16")
    if picked != report["source_video_frames"]:
        problems.append(f"{s}: the eval would show video frames {picked}, but the sheets were made on "
                        f"{report['source_video_frames']}")
if problems:
    raise RuntimeError("The video frames don't match the sheets, so the sheets would describe other frames than the "
                       "model sees:\n  " + "\n  ".join(problems) +
                       "\nThe videos or the recomputed frame_indices.npy differ from the cluster's. Use the cluster's "
                       "data (DATA_SOURCE = \"bundle\") instead.")
print("OK: every scene's 16 frames are the ones its sheets were made on")

## Input check

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, f"{WORK}/src")
from decord import VideoReader
from mvp.benchmarks import scene_sheet as ss
from mvp.benchmarks.vsibench import load_vsibench
from mvp.paths import VGGT_DIR, VSI_DIR

IDS = [int(x) for x in Path(IDS_PATH).read_text().split()]
index = ss.load_index(Path(SHEETS_ROOT))
vsi = load_vsibench("full").set_index("id")
need = [VSI_DIR / "test.jsonl", Path(SHEETS_ROOT) / "index.csv"]
for q in IDS:
    row = index[q]
    pipe = Path(SHEETS_ROOT) / row["pipeline_sheet"]
    need += [Path(vsi.loc[q, "video_path"]), VGGT_DIR / row["scene"] / "frame_indices.npy",
             VGGT_DIR / row["scene"] / "meta.json", pipe / "question.json", pipe / "objects.csv", pipe / "cameras.csv",
             pipe.parents[1] / "scene" / "adapter_report.json"]
missing = sorted({str(p) for p in need if not p.exists()})
assert not missing, "missing files:\n" + "\n".join(missing)
assert all(vsi.loc[q, "question"] == index[q]["question"] for q in IDS), "index.csv and test.jsonl disagree on a question"
for q in IDS:  # every cameras.csv has one row per frame (load_cameras raises otherwise)
    ss.load_cameras(q, Path(SHEETS_ROOT), VGGT_DIR)
n_empty = sum(not ss.load_objects(q, "pipeline", Path(SHEETS_ROOT), vggt_root=VGGT_DIR) for q in IDS)
print(f"all {len(set(map(str, need)))} files present for {len(IDS)} questions; every cameras.csv has 16 frames; "
      f"{n_empty} questions have no objects in their pipeline sheet")

for scene in sorted({index[q]["scene"] for q in IDS}):
    video = Path(vsi[vsi["scene_name"] == scene.split("/")[1]]["video_path"].iloc[0])
    vr = VideoReader(str(video))
    fps = json.loads((VGGT_DIR / scene / "meta.json").read_text())["video_fps"]
    print(f"{scene}: {len(vr)} frames, {fps:.2f} fps, {len(vr) / fps:.0f} s, "
          f"{sum(index[q]['scene'] == scene for q in IDS)} questions")

## Prompts

The conditions must differ only in the scene information between the video and the question and, for the CoT ones,
in the instructions after it. Check 2: the prompts the eval builds from each condition's arguments (`--print-prompt`:
the processor and the videos, not the weights), all 200 for `narrative` and the first `PRINT_N` for the others. In
each, the video's `<t seconds>` labels are the sheets' timestamps, every time a sheet writes is one of them, and the
prompt is `build_sheet_prompt`'s with the condition's options. Check 3, for all 200 questions in every condition:
- the text after the video is lmms-eval's with the sheet after its first line;
- every sheet is in metres;
- `cam_<f>` is `obj_<f>` with the cameras after the objects;
- the turn-then-move camera text changes only the camera path;
- each CoT condition is its direct one with only the last line, "answer directly", replaced by the CoT instructions.

In [ ]:
# Check 2: the eval's own prompts, from the arguments each condition runs with.
import os, re, subprocess
from mvp.paths import VGGT_DIR

HEADER = re.compile(r"^===== question (\d+) \((\S+)\), (\S+), video fps (\S+), frames (\[.*\])$", re.M)  # print_prompts'
LABEL = re.compile(r"<(\d+(?:\.\d+)?) seconds>")       # the processor's label before each pair of frames
FRAME_TOKENS = re.compile(re.escape(ss.VIDEO_PAD) + r" x (\d+)")
LINE_TIME = re.compile(r"^At (\d+(?:\.\d+)?) seconds:", re.M)  # a line of the camera text or the narrative
PRINT_N = 8                          # prompts printed per condition (all of them for FULL_CHECK)
FULL_CHECK = "narrative"
ROOT = Path(SHEETS_ROOT)


def scale_arg(name):
    return METRIC_SCALE if opts(name) else None


def sheet(q, name):
    """The scene information the condition puts in question q's prompt (None: video only)."""
    o = opts(name)
    if o is None:
        return None
    return ss.render_scene(q, "pipeline", o["fmt"], o["cameras"], ROOT, VGGT_DIR, o["time_unit"], o["visibility"],
                           o["step_order"], o["object_directions"], METRIC_SCALE)


def expected_prompt(q, name, frame_tokens=None):
    """build_sheet_prompt with the condition's options: the whole text input the eval should send."""
    o = opts(name)
    if o is None:
        return ss.build_sheet_prompt(q, None, None, ROOT, VGGT_DIR, frame_tokens=frame_tokens,
                                     reasoning=reasoning(name))
    return ss.build_sheet_prompt(q, "pipeline", o["fmt"], ROOT, VGGT_DIR, frame_tokens=frame_tokens, cameras=o["cameras"],
                                 time_unit=o["time_unit"], reasoning=reasoning(name), visibility=o["visibility"],
                                 step_order=o["step_order"], object_directions=o["object_directions"],
                                 metric_scale=METRIC_SCALE)


def sheet_times(q, name):
    """Every time, in seconds, the condition's sheet writes: the objects' visible ranges, the camera table's time
    column and the lines of the camera text or the narrative."""
    o = opts(name)
    times = set(LINE_TIME.findall(sheet(q, name)))
    if o["fmt"] != "narrative":
        objects = ss.load_objects(q, "pipeline", ROOT, vggt_root=VGGT_DIR, visibility=o["visibility"])
        times |= {t for obj in objects for pair in ss.ranges_of(obj["visible"]) for t in pair}
    if o["cameras"] and o["fmt"] != "text":
        times |= {c["time"] for c in ss.load_cameras(q, ROOT, VGGT_DIR)}
    return times


problems, checked = [], {}
for name in CONDITIONS:
    n = len(IDS) if name == FULL_CHECK else PRINT_N
    cmd = [sys.executable, "-m", "mvp.eval.vsibench", *eval_args(name), "--ids", IDS_PATH, "--print-prompt", str(n)]
    printed = subprocess.run(cmd, cwd=WORK, env={**os.environ, "PYTHONPATH": f"{WORK}/src"}, capture_output=True, text=True)
    assert printed.returncode == 0, f"{name}: " + printed.stdout[-3000:] + printed.stderr[-3000:]
    heads = list(HEADER.finditer(printed.stdout))
    checked[name] = len(heads)
    if len(heads) != n:
        problems.append(f"{name}: {len(heads)} prompts printed, not {n}")
    for head, nxt in zip(heads, heads[1:] + [None]):
        q = int(head.group(1))
        block = printed.stdout[head.end():nxt.start() if nxt else len(printed.stdout)]
        labels = LABEL.findall(block)
        row = index[q]
        expected = list(ss.frame_timestamps((ROOT / row["pipeline_sheet"]).parents[1] / "scene", row["scene"],
                                            VGGT_DIR)[::ss.MERGE])
        if labels != expected:
            problems.append(f"{name}, question {q}: the video's labels {labels} are not the sheet's timestamps {expected}")
        o = opts(name)
        if o and o["time_unit"] == "seconds" and not sheet_times(q, name) <= set(labels):
            problems.append(f"{name}, question {q}: times {sorted(sheet_times(q, name) - set(labels))} are not labels")
        tokens = FRAME_TOKENS.search(block)
        if block.strip("\n") != expected_prompt(q, name, tokens.group(1) if tokens else None).strip("\n"):
            problems.append(f"{name}, question {q}: the prompt is not build_sheet_prompt's with the condition's options")
if problems:
    raise RuntimeError("The eval's prompts are not as they should be:\n  " + "\n  ".join(problems[:20]))
print(f"OK: {sum(checked.values())} prompts ({', '.join(f'{k} {v}' for k, v in checked.items())}): the video's labels "
      "are the sheets' timestamps, every time a sheet writes is one of them, and each prompt is build_sheet_prompt's "
      "with the condition's options")

In [ ]:
# Check 3: what the conditions differ in, question by question, in the text the eval puts after the video (its own
# mvp.eval.vsibench.prompt_parts, with the eval's defaults for everything this notebook doesn't set).
import argparse
from mvp.benchmarks import vsibench as vb
from mvp.benchmarks.vsibench import load_vsibench
from mvp.eval import vsibench as ev

ORDER = load_vsibench("full")
ORDER = ORDER[ORDER["id"].isin(IDS)]  # the eval's order: test.jsonl's, questions grouped by video as it runs them
DOCS = {d["id"]: d for d in ORDER.to_dict("records")}
TYPE = {q: DOCS[q]["question_type"] for q in IDS}
KIND = ss.sheet_kind("pipeline", METRIC_SCALE)  # what the CoT instructions are told about the sheet: "pipeline_priors"
UNITS = re.compile(r"\bunits\b|relative units")  # a length left in the reconstruction's units


def namespace(name):
    """The eval's arguments prompt_parts reads: this condition's sheet options, the eval's defaults otherwise."""
    o = opts(name)
    options = {k: (o or DEFAULTS)[k] for k in DEFAULTS}  # cameras, time_unit, visibility, step_order, object_directions
    return argparse.Namespace(sheets=o is not None, sheet_source="pipeline" if o else None,
                              sheet_format=o["fmt"] if o else None, sheets_root=SHEETS_ROOT, vggt_root=str(VGGT_DIR),
                              metric_scale=scale_arg(name), layout="sit", reasoning=reasoning(name), **options)


TEXT = {name: {q: ev.prompt_parts(DOCS[q], namespace(name)) for q in IDS} for name in CONDITIONS}
SHEET = {name: {q: sheet(q, name) for q in IDS} for name in CONDITIONS}
problems = []
for q in IDS:
    d = DOCS[q]
    for name in CONDITIONS:
        pre, post = TEXT[name][q]
        if pre:
            problems.append(f"question {q}, {name}: text before the video")
        if post != vb.build_prompt(d, SHEET[name][q], reasoning(name), KIND if opts(name) else None):
            problems.append(f"question {q}, {name}: not lmms-eval's prompt with the sheet after its first line")
        if post not in expected_prompt(q, name):
            problems.append(f"question {q}, {name}: the text is not build_sheet_prompt's")
        if opts(name) and UNITS.search(SHEET[name][q]):
            problems.append(f"question {q}, {name}: the sheet is not in metres ({UNITS.search(SHEET[name][q]).group(0)!r})")
    for f in ("csv", "markdown_table", "text"):
        if SHEET[f"cam_{f}"][q] != SHEET[f"obj_{f}"][q] + "\n" + ss.render_cameras(q, f, ROOT, VGGT_DIR,
                                                                                    metric_scale=METRIC_SCALE):
            problems.append(f"question {q}: cam_{f} is not obj_{f} with the cameras after the objects")
    if SHEET["cam_text_turn_move"][q].split("\nCamera path:")[0] != SHEET["cam_text"][q].split("\nCamera path:")[0]:
        problems.append(f"question {q}: cam_text_turn_move changes more than the camera path")
    for cot, base in COT_BASES.items():
        body, last = TEXT[base][q][1].rsplit("\n", 1)
        if (last not in (vb.MCA_POST_PROMPT, vb.NA_POST_PROMPT)
                or TEXT[cot][q][1] != body + "\n" + vb.cot_prompt(d, True, KIND)):
            problems.append(f"question {q}: {cot} is not {base} with the CoT instructions as its last line")
if problems:
    raise RuntimeError("The conditions differ in more than they should:\n  " + "\n  ".join(problems[:20]))
n_empty = sum(ss.EMPTY in SHEET["obj_csv"][q] for q in IDS)
fallback = sum(ss.scene_scale.question_scale(q, str(ROOT))["fallback"] for q in IDS)
print(f"OK, for all {len(IDS)} questions in all {len(CONDITIONS)} conditions: only the scene information differs (and "
      f"the CoT instructions), and every sheet is in metres. {n_empty} questions have no objects (their sheets say so, "
      f"and still carry the cameras); {fallback} get the fallback scale ({ss.scene_scale.FALLBACK} m per unit), having "
      "no object of a known typical size")

In [ ]:
# One question's scene information in every condition, after the whole cam_csv prompt (chat template, the video's
# "<t seconds>" labels with one <|video_pad|> per frame pair, the sheet, the question), and the CoT instructions.
# Change Q to see another.
Q = next(q for q in IDS if TYPE[q].startswith("object_rel_direction") and ss.EMPTY not in SHEET["obj_csv"][q])
print(f"question {Q} ({TYPE[Q]}): ground truth {DOCS[Q]['ground_truth']}; "
      f"{ss.scale_of(Q, ROOT, METRIC_SCALE):.3f} m per sheet unit\n")
print("===== cam_csv, the whole prompt =====")
print(expected_prompt(Q, "cam_csv"))
for name in DIRECT:
    if name not in ("video", "cam_csv"):
        print(f"\n===== {name}: the scene information =====")
        print(SHEET[name][Q])
print("\n===== the CoT conditions: the last line, in place of lmms-eval's =====")
print(vb.cot_prompt(DOCS[Q], True, KIND))

## Input lengths

Every question's input length in every condition, as the eval's batches have it (the processor only, no weights). They
show what each sheet adds, give the batches for the time estimate, and decide whether the video-only predictions on
Drive were made with these prompts (Runs).

In [ ]:
import pandas as pd
from mvp.models.qwen_vl import QwenVL

PROMPTER = QwenVL(MODEL, weights=False)  # the processor and the tokenizer, without the weights


def chat_tokens(name, q):
    """The tokens of the condition's chat text for question q with the video as its one placeholder. The processor
    expands only the placeholder, between special tokens the text is split at, so a condition's input length is the
    video-only one plus this count's difference from the video-only text's."""
    pre, post = TEXT[name][q]
    text = PROMPTER.processor.apply_chat_template(PROMPTER._messages(post, "video", pre=pre), tokenize=False,
                                                  add_generation_prompt=True, **PROMPTER.template_kwargs)
    return len(PROMPTER.processor.tokenizer(text, add_special_tokens=False)["input_ids"])


def processor_lengths(name, docs, video, meta, vkw):
    """Input lengths as the eval's batch has them: the processor on the video and the prompts."""
    parts = [TEXT[name][d["id"]] for d in docs]
    mask = PROMPTER.video_inputs(video, meta, vkw, [p for _, p in parts], pres=[q for q, _ in parts])["attention_mask"]
    return {d["id"]: int(n) for d, n in zip(docs, mask.sum(1))}


LENGTHS = {name: {} for name in CONDITIONS}
VIDEO_OF, FRAMES, VIS_TOKENS = {}, {}, {}
for k, (path, group) in enumerate(ORDER.groupby("video_path", sort=False)):
    video, meta, vkw = PROMPTER.load_video_frames(path, ev.frame_indices(path, "uniform128", 16, str(VGGT_DIR)))
    VIS_TOKENS[path] = video.shape[0] * video.shape[2] * video.shape[3] // (2 * 32 * 32)  # as the eval counts them
    docs = group.to_dict("records")
    for d in docs:
        VIDEO_OF[d["id"]], FRAMES[d["id"]] = path, list(meta["frames_indices"])
    LENGTHS["video"].update(processor_lengths("video", docs, video, meta, vkw))
    for name in CONDITIONS:
        LENGTHS[name].update({d["id"]: LENGTHS["video"][d["id"]] + chat_tokens(name, d["id"]) - chat_tokens("video", d["id"])
                              for d in docs})
        if k == 0:  # on the first video, check the shortcut against the processor in every condition
            assert processor_lengths(name, docs, video, meta, vkw) == {d["id"]: LENGTHS[name][d["id"]] for d in docs}, name
lengths = pd.DataFrame(LENGTHS).loc[IDS]
SHEET_TOKENS = lengths.sub(lengths["video"], axis=0)  # what the scene information adds to the video-only prompt
print(f"input tokens per question ({len(IDS)} questions; the video is {min(VIS_TOKENS.values())} to "
      f"{max(VIS_TOKENS.values())} of them), and what the scene information adds:")
display(pd.DataFrame({"family": {n: family(n) for n in CONDITIONS}, "input mean": lengths.mean(),
                      "input max": lengths.max(), "sheet mean": SHEET_TOKENS.mean(),
                      "sheet max": SHEET_TOKENS.max()}).round(0))

## Runs

First `video`, copied from Drive if its predictions check out. Then 5 questions of one video are timed in every
condition: a direct one after a warm-up question, and each of the four CoT candidates. The estimate for the rest then
decides the CoT questions: all 200 or, with `SUBSET = k`, k questions of every question type (all of a type that has
fewer: relative direction has 8, 8 and 9), taken scene by scene in a random order of the scenes (seed 0), since the time
grows with the number of batches. The direct conditions run first, on all 200; then the table format for CoT is chosen;
then the three CoT conditions run.

In [ ]:
import json, math, os, subprocess, sys, threading, time
from collections import Counter
from pathlib import Path

os.makedirs(RESULTS_DIR, exist_ok=True)
BY_VIDEO = ORDER.groupby("video_path", sort=False)["id"].apply(list)  # the eval's order of videos and questions


def out(name):
    """The condition's predictions."""
    return Path(RESULTS_DIR) / f"{name}.jsonl"


def answered(path):
    """The question ids a prediction file already answers. A crash can leave a half-written last line, which would stop
    the eval's own resume (it reads every line as JSON): unreadable lines are dropped from the file first."""
    path = Path(path)
    if not path.exists():
        return set()
    lines = path.read_text().splitlines()
    good = []
    for line in lines:
        try:
            good.append(json.loads(line))
        except json.JSONDecodeError:
            pass
    if len(good) != len(lines):
        print(f"{path.name}: dropped {len(lines) - len(good)} unreadable line(s) left by a crash")
        path.write_text("".join(json.dumps(r) + "\n" for r in good))
    return {r["id"] for r in good}


def count(path, ids):
    """How many of ids a prediction file answers, reading it as it is written (for the progress lines)."""
    try:
        lines = Path(path).read_text().splitlines()
    except FileNotFoundError:
        return 0
    got = set()
    for line in lines:
        try:
            got.add(json.loads(line)["id"])
        except (json.JSONDecodeError, KeyError):
            pass
    return len(got & set(ids))


def run_eval(name, ids_path, out_path, every=300, watch=None):
    """The eval for one condition on the questions of ids_path (it skips those out_path already answers), with its
    output and, every `every` seconds, how many are answered. watch, a list: (seconds since the start, questions
    answered) is appended to it each time out_path gains answers (the eval writes them batch by batch). Returns
    (seconds, the eval's output)."""
    ids = [int(x) for x in Path(ids_path).read_text().split()]
    cmd = [sys.executable, "-m", "mvp.eval.vsibench", *eval_args(name), "--ids", str(ids_path), "--out", str(out_path)]
    print(f"===== {name}: {' '.join(cmd[1:])}", flush=True)
    t0, stop = time.time(), threading.Event()

    def progress():
        while not stop.wait(every):
            print(f"  [{name}: {count(out_path, ids)} of {len(ids)} answered, {(time.time() - t0) / 60:.0f} min]", flush=True)

    def watcher():
        last = count(out_path, ids)
        while not stop.wait(0.2):
            n = count(out_path, ids)
            if n != last:
                watch.append((time.time() - t0, n))
                last = n

    threading.Thread(target=progress, daemon=True).start()
    if watch is not None:
        threading.Thread(target=watcher, daemon=True).start()
    proc = subprocess.Popen(cmd, cwd=WORK, env={**os.environ, "PYTHONPATH": f"{WORK}/src"}, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    stop.set()
    assert proc.wait() == 0, f"{name} failed"
    print(f"===== {name} done in {time.time() - t0:.0f} s\n", flush=True)
    return time.time() - t0, "".join(lines)


def batch_size(video):
    """The eval's batch size for a video: min(BATCH_SIZE, TOKEN_BUDGET // (visual tokens + 200))."""
    return max(1, min(BATCH_SIZE, TOKEN_BUDGET // (VIS_TOKENS[video] + 200)))


def batches(name, ids):
    """The batches the eval will make of the questions of ids that `name` has not answered, in its order."""
    done, ids = answered(out(name)), set(ids)
    result = []
    for video, qs in BY_VIDEO.items():
        left, bs = [q for q in qs if q in ids and q not in done], batch_size(video)
        result += [left[i:i + bs] for i in range(0, len(left), bs)]
    return result


def padded(name, batch):
    """A batch's input tokens as the model gets them: left-padded to its longest prompt."""
    return max(LENGTHS[name][q] for q in batch) * len(batch)

In [ ]:
# video: copied from Drive when the predictions answer the same questions, on the same frames, scored by the
# final-answer rule, with replies shorter than the token limit, and with the prompts and batches this code makes: the
# eval records each batch's padded input length (n_input_tokens), so every batch this notebook would make must have one
# recorded length there, its longest prompt's as built here. Copied predictions are listed in reused.json.
from mvp.models.qwen_vl import SYSTEM_PROMPT


def reuse_problems(path):
    """Why the predictions in `path` cannot stand in for video (empty: they can)."""
    path = Path(path)
    if not path.exists():
        return [f"no {path}"]
    run = pd.read_json(path, lines=True)
    if len(run) != len(IDS) or set(run["id"]) != set(IDS):
        return [f"{run['id'].nunique()} questions in {len(run)} lines, not these {len(IDS)}"]
    problems = []
    summary = path.with_name(path.stem + "_summary.json")
    if summary.exists():  # the run's settings, where its summary has them
        s = json.loads(summary.read_text())
        want = {"model": MODEL, "nframes": 16, "sampling": "uniform128", "layout": "sit", "system": SYSTEM_PROMPT,
                "final_answer": True, "reasoning": "direct", "scene_sheet": None}
        problems += [f"its summary has {k} = {s[k]!r}, not {v!r}" for k, v in want.items() if k in s and s[k] != v]
    if "score_first" not in run:
        problems.append("not scored with --final-answer")
    other = [q for q, f in zip(run["id"], run["frame_indices"]) if list(f) != FRAMES[q]]
    if other:
        problems.append(f"{len(other)} questions on other frames than the eval picks here")
    recorded = dict(zip(run["id"], run["n_input_tokens"]))
    bad = []
    for video, qs in BY_VIDEO.items():  # the batches this notebook makes: each must have been one batch there
        bs = batch_size(video)
        for b in (qs[i:i + bs] for i in range(0, len(qs), bs)):
            there, here = {recorded[q] for q in b}, max(LENGTHS["video"][q] for q in b)
            if there != {here}:
                bad.append(f"questions {b}: {sorted(there)} input tokens there, {here} here")
    if bad:
        problems.append(f"{len(bad)} batches had other prompts or other batches, e.g. " + "; ".join(bad[:2]))
    longest_reply = max(map(len, PROMPTER.processor.tokenizer(list(run["prediction"].astype(str)),
                                                               add_special_tokens=False)["input_ids"]))
    if longest_reply >= DIRECT_TOKENS:
        problems.append(f"a reply of {longest_reply} tokens, which the limit of {DIRECT_TOKENS} would have cut off")
    return problems


REUSED_JSON = Path(RESULTS_DIR) / "reused.json"
reused = json.loads(REUSED_JSON.read_text()) if REUSED_JSON.exists() else {}
have = answered(out("video")) & set(IDS)
if len(have) == len(IDS):
    print(f"video: all {len(IDS)} answered" + (f", copied from {reused['video']}" if "video" in reused else ""))
elif have:
    print(f"video: {len(have)} of {len(IDS)} answered here; the rest run with the others")
elif not REUSE:
    print("video: REUSE is False; it runs with the others")
else:
    for src in REUSE_VIDEO:
        problems = reuse_problems(src)
        print(f"video from {src}: " + ("; ".join(problems) if problems else "same questions, frames, prompts and rule"))
        if not problems:
            shutil.copy(src, out("video"))
            reused["video"] = src
            break
    else:
        print("video: none of REUSE_VIDEO checks out; it runs with the others")
    REUSED_JSON.write_text(json.dumps(reused, indent=1))

In [ ]:
# Time TIMING_N questions per condition: the questions of one video, the video with exactly TIMING_N questions and the
# most question types (the first in the eval's order among equals), so that the batch is the one the full run would
# make.
#   - Direct: the eval's first batch also pays for warming up the GPU, which the full run pays once, so each timing run
#     starts with one question of the first video (WARM_UP) and ends with one of the last (TAIL, so that the timed batch
#     is never the eval's last write); the timed batch's time is from the warm-up's answers being written to its own.
#   - CoT: only the timed questions (a warm-up and a tail would each cost up to 2,048 decoding steps); the batch's time
#     is from the start of the eval to its answers being written, less the direct condition's start-up and warm-up.
# They run into a scratch file; the timed answers are then added to the condition's predictions, so the full run
# doesn't redo them, and timing.json keeps the times, so a resumed notebook doesn't time again on the same GPU.
WARM_UP, TAIL = BY_VIDEO.iloc[0][0], BY_VIDEO.iloc[-1][0]
inner = list(BY_VIDEO.index[1:-1])
candidates = ([v for v in inner if len(BY_VIDEO[v]) == TIMING_N] or [v for v in inner if len(BY_VIDEO[v]) > TIMING_N]
              or inner)
TIMING_VIDEO = max(candidates, key=lambda v: len({TYPE[q] for q in BY_VIDEO[v][:TIMING_N]}))
TIMING_IDS = BY_VIDEO[TIMING_VIDEO][:TIMING_N]
TIMING_DIR = Path(RESULTS_DIR) / "timing"
TIMING_DIR.mkdir(exist_ok=True)
(TIMING_DIR / "timing_ids.txt").write_text("".join(f"{q}\n" for q in [WARM_UP, *TIMING_IDS, TAIL]))
(TIMING_DIR / "timing_ids_cot.txt").write_text("".join(f"{q}\n" for q in TIMING_IDS))
print(f"timed: {TIMING_VIDEO.split('VSI-Bench/')[-1]}, questions {TIMING_IDS} ({', '.join(TYPE[q] for q in TIMING_IDS)}); "
      f"the direct conditions after warm-up question {WARM_UP} and before tail question {TAIL}")

TIMING_JSON = Path(RESULTS_DIR) / "timing.json"
COT_TABLE_JSON = Path(RESULTS_DIR) / "cot_table.json"
timing = json.loads(TIMING_JSON.read_text()) if TIMING_JSON.exists() else {}
chosen = json.loads(COT_TABLE_JSON.read_text())["table"] if COT_TABLE_JSON.exists() else COT_TABLE
for name in CONDITIONS:
    setup = {"gpu": GPU, "max_new_tokens": max_tokens(name), "batch_size": BATCH_SIZE, "questions": TIMING_IDS}
    if answered(out(name)) >= set(IDS):
        print(f"{name}: all {len(IDS)} questions answered, nothing to time")
        continue
    if chosen and name in TABLE_COT.values() and name != TABLE_COT[chosen]:
        print(f"{name}: not run (the CoT table format is {chosen}), nothing to time")
        continue
    if "batch_s" in timing.get(name, {}) and all(timing[name].get(k) == v for k, v in setup.items()):
        continue  # timed before, with these settings on this GPU
    cot = name in COT_BASES
    scratch = TIMING_DIR / f"{name}.jsonl"
    scratch.unlink(missing_ok=True)
    watch = []
    wall, _ = run_eval(name, TIMING_DIR / ("timing_ids_cot.txt" if cot else "timing_ids.txt"), scratch, watch=watch)
    when = {n: t for t, n in reversed(watch)}  # questions answered -> the first time the file had that many
    recs = {r["id"]: r for r in map(json.loads, scratch.read_text().splitlines())}
    bs = batch_size(TIMING_VIDEO)  # one batch unless the batch size is under TIMING_N
    chunks = [TIMING_IDS[i:i + bs] for i in range(0, len(TIMING_IDS), bs)]
    tokens = [recs[q]["n_new_tokens"] for q in TIMING_IDS]
    if cot:
        base = timing.get(COT_BASES[name]) or {}
        start = base.get("start_s", min([u["start_s"] for u in timing.values() if "start_s" in u] or [20.0]))
        done = when.get(len(TIMING_IDS), wall)  # the eval's last write can come just before it exits
        batch_s = done - start
    else:
        n_timed = 1 + len(TIMING_IDS)
        assert 1 in when and n_timed in when, f"the watcher missed a batch ({watch}): run this cell again"
        start, batch_s = when[1], when[n_timed] - when[1]
    timing[name] = {**setup, "batch_s": round(batch_s, 1), "start_s": round(start, 1), "wall_s": round(wall, 1),
                    "reply_tokens": tokens, "hit_token_limit": [recs[q]["hit_token_limit"] for q in TIMING_IDS],
                    "steps": sum(max(tokens[TIMING_IDS.index(q)] for q in c) for c in chunks),  # each batch's longest
                    "padded_tokens": sum(recs[c[0]]["n_input_tokens"] * len(c) for c in chunks)}  # the eval's padded length
    have = answered(out(name))
    with open(out(name), "a") as f:
        f.writelines(json.dumps(recs[q]) + "\n" for q in TIMING_IDS if q not in have)
    TIMING_JSON.write_text(json.dumps(timing, indent=1))
for name, t in timing.items():
    print(f"{name}: {len(t['questions'])} questions, {t['padded_tokens']:,} input tokens, in {t['batch_s']:.1f} s on "
          f"{t['gpu']} ({t['start_s']:.0f} s to start the eval and warm up); replies of {t['reply_tokens']} tokens, "
          f"{sum(t['hit_token_limit'])} cut off at {t['max_new_tokens']}")

In [ ]:
# The estimate per condition.
#   - Direct: replies are a few tokens, so a batch's time is mostly reading its input, and each batch left takes the
#     timed batch's time scaled by its input tokens (padded, as the model gets them).
#   - CoT: a batch runs until its longest reply ends. Its time is the reading of its input, at the direct condition's
#     rate, plus a decoding step's time (from the timed batch, less its reading) for each step of its longest reply,
#     expected from the timed replies' lengths. "If every CoT batch hit the limit" is the upper bound.
# The timed time includes loading the video, which the full run does once per video, so the estimate is a little high.
# Each run also starts the eval and warms up once (start_s). The CoT table format isn't chosen yet, so the totals count
# the slower of the two.
import random


def per_type(k, seed=0):
    """k questions of every question type (all of a type that has fewer), taken scene by scene in a random order of
    the scenes, so that they come from few videos."""
    videos = list(BY_VIDEO.index)
    random.Random(seed).shuffle(videos)
    need = Counter({t: min(k, n) for t, n in Counter(TYPE.values()).items()})
    chosen_ids = []
    for v in videos:
        for q in BY_VIDEO[v]:
            if need[TYPE[q]] > 0:
                chosen_ids.append(q)
                need[TYPE[q]] -= 1
        if not +need:
            break
    return sorted(chosen_ids)


def expected_longest(lengths, b):
    """The expected longest of b replies drawn at random, with replacement, from `lengths`."""
    L, n = sorted(lengths), len(lengths)
    return sum(L[k] * (((k + 1) / n) ** b - (k / n) ** b) for k in range(n))


def estimate(name, ids, worst=False):
    """Expected seconds for `name` on the questions of ids it has not answered (worst: every CoT batch to the limit)."""
    left = batches(name, ids)
    if not left:
        return 0.0
    t = timing[name]
    if name not in COT_BASES:
        return t["start_s"] + t["batch_s"] / t["padded_tokens"] * sum(padded(name, b) for b in left)
    base = timing.get(COT_BASES[name], t)
    rate = base["batch_s"] / base["padded_tokens"] if base is not t else 0.0  # reading, s per input token
    step = max(t["batch_s"] - rate * t["padded_tokens"], 1.0) / max(t["steps"], 1)
    return t["start_s"] + sum(rate * padded(name, b) + step * (COT_TOKENS if worst else
                                                               expected_longest(t["reply_tokens"], len(b))) for b in left)


if SUBSET not in (None, "all") and not isinstance(SUBSET, int):
    raise ValueError(f"SUBSET must be None, 'all' or a number of questions per question type, not {SUBSET!r}")
TABLES = [TABLE_COT[chosen]] if chosen else list(TABLE_COT.values())
ESTIMATED = list(DIRECT) + TABLES + ["cot_cam_text", "cot_narrative"]
missing = [n for n in ESTIMATED if batches(n, IDS) and n not in timing]
assert not missing, f"no timing for {missing}: run the cell above"
choices = {"all": IDS, **{k: per_type(k) for k in sorted({10, 5} | ({SUBSET} if isinstance(SUBSET, int) else set()),
                                                          reverse=True)}}


def total(ids, worst=False):
    """Hours left: the direct conditions on all 200, the CoT ones on ids (the slower table format)."""
    s = sum(estimate(n, IDS) for n in DIRECT)
    s += max(estimate(n, ids, worst) for n in TABLES) + sum(estimate(n, ids, worst) for n in ("cot_cam_text", "cot_narrative"))
    return s / 3600


est = pd.DataFrame({("all" if key == "all" else f"SUBSET = {key!r}"):
                    {n: estimate(n, IDS if n in DIRECT else ids) / 60 for n in ESTIMATED} for key, ids in choices.items()})
rows = pd.DataFrame([{c: total(ids) for c, ids in zip(est.columns, choices.values())},
                     {c: total(ids, worst=True) for c, ids in zip(est.columns, choices.values())}],
                    index=["all conditions (h)", "if every CoT batch hit the limit (h)"])
print(f"Estimated minutes left per condition on this GPU ({GPU}; the direct ones always on all 200), and in all:")
display(pd.concat([est.round(0), rows.round(1)]))
print("CoT questions:", {key: len(ids) for key, ids in choices.items()})
left_all = total(IDS)
if SUBSET is None and left_all > HOURS_LIMIT:
    raise RuntimeError(
        f"The runs would take about {left_all:.1f} h more on this GPU with the CoT conditions on all {len(IDS)} "
        f"questions, over HOURS_LIMIT = {HOURS_LIMIT} h. Choose in the settings cell (Paths): SUBSET = 10 or SUBSET = 5 "
        f"(CoT questions per question type; {', '.join(f'{k}: {total(ids):.1f} h' for k, ids in list(choices.items())[1:])}), "
        "or SUBSET = \"all\" to run all of them anyway; then run the notebook again from the top. Nothing done so far "
        "is redone, and the timing is kept.")
CHOICE = "all" if SUBSET in (None, "all") else SUBSET
RUN_IDS = choices[CHOICE]
RUN_IDS_PATH = Path(RESULTS_DIR) / "run_ids_cot.txt"
RUN_IDS_PATH.write_text("".join(f"{q}\n" for q in RUN_IDS))
print(f"The runs: the direct conditions on all {len(IDS)} questions, the CoT ones on {len(RUN_IDS)} (SUBSET = "
      f"{CHOICE!r}), about {total(RUN_IDS):.1f} h left; CoT questions per question type: "
      f"{dict(Counter(TYPE[q] for q in RUN_IDS))}")

In [ ]:
# The direct conditions, on all 200 questions.
t_runs = time.time()


def left_minutes():
    return (sum(estimate(n, IDS) for n in DIRECT) + max(estimate(n, RUN_IDS) for n in TABLES)
            + sum(estimate(n, RUN_IDS) for n in ("cot_cam_text", "cot_narrative"))) / 60


for name in DIRECT:
    done = answered(out(name)) & set(IDS)
    if len(done) == len(IDS):
        print(f"===== {name}: all {len(IDS)} questions answered, skipped", flush=True)
        continue
    if done:
        print(f"===== {name}: resuming, {len(done)} of {len(IDS)} answered", flush=True)
    run_eval(name, IDS_PATH, out(name))
    print(f"{(time.time() - t_runs) / 60:.0f} min since the runs started; about {left_minutes():.0f} min left\n", flush=True)

In [ ]:
# The table format for CoT: the one whose direct cam_<format> has the higher overall (VSI-Bench's 8-task mean) on all
# 200 questions here, csv on a tie (it is the shorter), unless COT_TABLE chooses it by hand. The choice is kept in
# cot_table.json, so a resumed notebook keeps it.
from mvp.benchmarks.vsibench import summarize

COT_TABLE_JSON = Path(RESULTS_DIR) / "cot_table.json"


def overall(name):
    run = pd.read_json(out(name), lines=True).drop_duplicates("id").set_index("id").loc[IDS]
    return summarize(run[["question_type", "score"]])["overall"]


if COT_TABLE_JSON.exists():
    pick = json.loads(COT_TABLE_JSON.read_text())
    print(f"the CoT table format, chosen before: {pick}")
else:
    scores_by_format = {f: overall(f"cam_{f}") for f in TABLE_COT}
    best = COT_TABLE or ("markdown_table" if scores_by_format["markdown_table"] > scores_by_format["csv"] else "csv")
    pick = {"table": best, "rule": "by hand (COT_TABLE)" if COT_TABLE else "the higher direct overall, csv on a tie",
            "direct overall": scores_by_format}
    COT_TABLE_JSON.write_text(json.dumps(pick, indent=1))
    print(f"the CoT table format: {pick}")
COT_RUN = [TABLE_COT[pick["table"]], "cot_cam_text", "cot_narrative"]
TABLES = [TABLE_COT[pick["table"]]]
print("the CoT conditions:", ", ".join(COT_RUN))

In [ ]:
# The CoT conditions, on RUN_IDS.
for name in COT_RUN:
    done = answered(out(name)) & set(RUN_IDS)
    if len(done) == len(RUN_IDS):
        print(f"===== {name}: all {len(RUN_IDS)} questions answered, skipped", flush=True)
        continue
    if done:
        print(f"===== {name}: resuming, {len(done)} of {len(RUN_IDS)} answered", flush=True)
    run_eval(name, RUN_IDS_PATH, out(name))
    print(f"{(time.time() - t_runs) / 60:.0f} min since the runs started; about {left_minutes():.0f} min left\n", flush=True)

## Answers: one rule, parse rate, cut-off replies

Every reply is read by one rule, the eval's final-answer rule (`score_final`), applied again here to the copied
predictions too: for multiple choice, the letter of the last "Answer: X" or "answer is X", else the last letter that
stands alone in the reply's last line; for numbers, the last number in the reply. A reply it reads nothing from scores
0. A CoT reply cut off at the token limit has no answer line, but the rule may still read a letter or number from its
unfinished reasoning: those are counted apart, and "score, cut-off replies wrong" counts them as 0. Cut-off replies that
**loop** are counted as in the structured-CoT pilot: they narrate the video (10 or more mm:ss times) or repeat
themselves (their last 2,000 characters compress, with zlib, below a quarter of their size).

In [ ]:
import random
import re
import zlib

import numpy as np
from mvp.benchmarks.vsibench import (MCA_QUESTION_TYPES, METRIC_TASKS, TASKS, final_letter, final_number, parsed,
                                     score_final, summarize, task_of)

COT_TABLE_JSON = Path(RESULTS_DIR) / "cot_table.json"
COT_RUN = globals().get("COT_RUN") or [TABLE_COT[json.loads(COT_TABLE_JSON.read_text())["table"]], "cot_cam_text",
                                       "cot_narrative"]
RUN_IDS = globals().get("RUN_IDS", IDS)
ACTIVE = list(DIRECT) + COT_RUN
QUESTIONS = {n: (RUN_IDS if n in COT_BASES else IDS) for n in ACTIVE}  # the questions each condition ran on
ANSWER_LINE = re.compile(r"^[\s*#>_-]*answer\s*\**\s*:", re.I)
FRAME_TIME = re.compile(r"(?<![\d:])\d{1,2}:\d{2}(?![\d:])")  # mm:ss, as the structured-CoT pilot counts them
NARRATES, REPEATS = 10, 0.25


def read(d, reply):
    """The answer the rule reads from a reply: a letter, a number, or None."""
    return final_letter(reply) if d["question_type"] in MCA_QUESTION_TYPES else final_number(reply)


def answer_line(reply):
    lines = [line for line in reply.splitlines() if ANSWER_LINE.match(line)]
    return lines[-1] if lines else None


def loops(reply):
    tail = reply[-2000:].encode()
    return len(FRAME_TIME.findall(reply)) >= NARRATES or len(zlib.compress(tail)) / max(1, len(tail)) < REPEATS


runs = {}
for name in ACTIVE:
    if not out(name).exists():
        continue
    run = pd.read_json(out(name), lines=True).drop_duplicates("id").set_index("id")
    run = run.loc[[q for q in QUESTIONS[name] if q in run.index]].copy()
    run["prediction"] = run["prediction"].astype(str)
    docs = [DOCS[q] for q in run.index]
    run["rescored"] = [score_final(d, p) for d, p in zip(docs, run["prediction"])]
    run["read"] = pd.Series([read(d, p) for d, p in zip(docs, run["prediction"])], index=run.index, dtype=object)
    run["parsed"] = [parsed(d, p, final=True) for d, p in zip(docs, run["prediction"])]
    run["line"] = pd.Series([answer_line(p) for p in run["prediction"]], index=run.index, dtype=object)
    for col in ("n_new_tokens", "hit_token_limit"):  # copied predictions made before the eval recorded them
        if col not in run:
            run[col] = None
    missing = run["n_new_tokens"].isna()
    if missing.any():
        counted = [len(t) for t in PROMPTER.processor.tokenizer(list(run.loc[missing, "prediction"]),
                                                                add_special_tokens=False)["input_ids"]]
        run.loc[missing, "n_new_tokens"] = counted
        run.loc[missing, "hit_token_limit"] = [n >= max_tokens(name) for n in counted]
    run["n_new_tokens"] = run["n_new_tokens"].astype(int)
    run["hit_token_limit"] = run["hit_token_limit"].astype(bool)
    runs[name] = run
absent = [n for n in ACTIVE if n not in runs or len(runs[n]) < len(QUESTIONS[n])]
print(f"conditions with all their predictions: {len(ACTIVE) - len(absent)} of {len(ACTIVE)}"
      + (f"; incomplete or missing: {', '.join(absent)}" if absent else ""))
differ = {n: int((abs(r["rescored"] - r["score"]) > 1e-9).sum()) for n, r in runs.items()}
print("scores here that differ from the eval's own (the same rule, so none should):", differ)

answers = {}
for name, run in runs.items():
    cut, has_line = run["hit_token_limit"], run["line"].notna()
    answers[name] = {
        "family": family(name), "replies": len(run), "parsed %": 100 * run["parsed"].mean(),
        "'Answer:' line %": 100 * has_line.mean(), "median tokens": run["n_new_tokens"].median(),
        "90th pct tokens": run["n_new_tokens"].quantile(0.9), f"hit the limit": int(cut.sum()),
        "of them before answering": int((cut & ~has_line).sum()),
        "of those, read anyway": int((cut & ~has_line & run["parsed"]).sum()),
        "cut off and looping": int((cut & run["prediction"].map(loops)).sum()),
        "input tokens, mean": lengths.loc[run.index, name].mean(),
        "mean score": 100 * run["rescored"].mean(),
        "score, cut-off replies wrong": 100 * run["rescored"].where(~(cut & ~has_line), 0).mean(),
    }
answers = pd.DataFrame.from_dict(answers, orient="index")
display(answers.round(1))
per_type_cot = pd.concat({n: runs[n].groupby("question_type").agg(
    **{"median tokens": ("n_new_tokens", "median"), "hit the limit": ("hit_token_limit", "sum"),
       "parsed %": ("parsed", lambda s: 100 * s.mean()), "n": ("parsed", "size")}) for n in COT_RUN if n in runs},
    axis=1) if any(n in runs for n in COT_RUN) else pd.DataFrame()
print("The CoT replies per question type:")
display(per_type_cot.round(0))

## Per-question table

In [ ]:
# Each condition's answer as the rule reads it, ✓/✗ (multiple choice) or the score (numbers), ✂ if cut off at the limit.
def cell(run, q):
    if q not in run.index:
        return "—"
    a, s = run.loc[q, "read"], run.loc[q, "rescored"]
    mark = " ✂" if run.loc[q, "hit_token_limit"] else ""
    if TYPE[q] in MCA_QUESTION_TYPES:
        return f"{a} {'✓' if s == 1 else '✗'}{mark}"
    return f"{'?' if pd.isna(a) else f'{a:g}'} ({s:.2f}){mark}"


table = pd.DataFrame({"scene": {q: DOCS[q]["dataset"] + "/" + DOCS[q]["scene_name"] for q in IDS},
                      "type": {q: TYPE[q] for q in IDS}, "truth": {q: DOCS[q]["ground_truth"] for q in IDS}})
for name, run in runs.items():
    table[name] = [cell(run, q) for q in table.index]
table.index.name = "id"
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 250)
display(table)

Full replies, for any question: `runs['cot_narrative'].loc[QUESTION_ID, 'prediction']`.

## Summary: the direct conditions

Score ×100 on all 200 questions, as in run 1. **Overall** is VSI-Bench's (the mean of the 8 task scores, relative
direction's three types as one task). **Metric tasks**: the mean of absolute distance, object size and room size, which
the metres are for. **Direction and route**: the mean of relative direction and route planning. **Without metric
tasks**: the other 5 tasks. Sheet tokens: what the scene information adds to the video-only prompt, on average.

In [ ]:
TYPE_ORDER = [t for k in TASKS for t in ([f"object_rel_direction_{x}" for x in ("easy", "medium", "hard")]
                                         if k == "object_rel_direction" else [k])]
GROUPS = {"overall": set(TASKS), "metric tasks": set(METRIC_TASKS),
          "direction and route": {"object_rel_direction", "route_planning"},
          "without metric tasks": set(TASKS) - set(METRIC_TASKS)}
LEAD = {"overall": 2, "direction and route": 5}
B = 10_000


def score_table(S, types, names):
    """Per condition: the groups, each task, and the sheet tokens (scores x100)."""
    def group_score(name, tasks):
        keep = types.index[types.map(task_of).isin(tasks)]
        return summarize(pd.DataFrame({"question_type": types[keep], "score": S.loc[keep, name]}))["overall"]
    out = pd.DataFrame({n: {"family": family(n.split(" ")[0]), **{g: group_score(n, ts) for g, ts in GROUPS.items()},
                            **{k: v for k, v in summarize(pd.DataFrame({"question_type": types, "score": S[n]})).items()
                               if k != "overall"},
                            "sheet tokens": SHEET_TOKENS.loc[types.index, n.split(" ")[0]].mean()} for n in names}).T
    for col in out.columns[1:]:
        out[col] = out[col].astype(float)
    return out


def scores_of(names, ids):
    return pd.DataFrame({n: runs[n].loc[ids, "rescored"] for n in names})


DIRECT_RUN = [n for n in DIRECT if n in runs]
D_IDS = [q for q in IDS if all(q in runs[n].index for n in DIRECT_RUN)]
S_direct = scores_of(DIRECT_RUN, D_IDS)
types_direct = pd.Series({q: TYPE[q] for q in D_IDS})
scores = score_table(S_direct, types_direct, DIRECT_RUN)
print(f"{len(D_IDS)} questions; per task: {dict(Counter(types_direct.map(task_of)))}")
display(scores.round(1))
qtypes = [t for t in TYPE_ORDER if t in set(types_direct)]
by_type = (100 * S_direct.groupby(types_direct).mean()).reindex(qtypes)
by_type.insert(0, "n", types_direct.value_counts().reindex(qtypes).astype(int))
display(by_type.round(1))

## Bootstrap intervals: the direct conditions

Each comparison as the difference and its 95% interval from 10,000 paired resamples: questions are drawn with
replacement within each question type, so every type keeps its number of questions, and all conditions get the same
draw. `*`: the interval excludes 0. **lead**: the README's reading rule, a difference of at least 5 points on direction
and route or 2 on the overall, with an interval that excludes 0. With about 25 comparisons, one or two will exclude 0
by chance, and with 25 questions a type a question type's interval is wide: a lead is something to confirm on more
questions, not a finding.

In [ ]:
def bootstrap(S, types, seed=0):
    """(question types, point: type x condition, draws: B x type x condition, conditions) for the scores S."""
    qt = [t for t in TYPE_ORDER if t in set(types)]
    X = S.to_numpy(float)
    members = [np.flatnonzero(types.to_numpy() == t) for t in qt]
    rng = np.random.default_rng(seed)
    point = np.stack([X[m].mean(0) for m in members])
    draws = np.stack([X[m][rng.integers(0, len(m), size=(B, len(m)))].mean(1) for m in members], axis=1)
    return qt, point, draws, list(S.columns)


def task_weights(qt, tasks):
    """w such that w @ (per question type means) is the mean of the scores of the tasks in `tasks`."""
    kept = [k for k in dict.fromkeys(task_of(t) for t in qt) if k in tasks]
    return np.mean([[(task_of(t) == k) / sum(task_of(u) == k for u in qt) for t in qt] for k in kept], axis=0)


def difference(boot, x, y):
    """{row: (difference, its bootstrap draws)} for x − y: the groups, then each question type."""
    qt, point, draws, conds = boot
    i, j = conds.index(x), conds.index(y)
    d_point, d_boot = 100 * (point[:, i] - point[:, j]), 100 * (draws[:, :, i] - draws[:, :, j])
    rows = {g: (d_point @ task_weights(qt, ts), d_boot @ task_weights(qt, ts)) for g, ts in GROUPS.items()
            if any(task_of(t) in ts for t in qt)}
    return {**rows, **{t: (d_point[k], d_boot[:, k]) for k, t in enumerate(qt)}}


def interval(p, b):
    lo, hi = np.percentile(b, [2.5, 97.5])
    return f"{p:+.1f} [{lo:+.1f}, {hi:+.1f}]" + (" *" if lo > 0 or hi < 0 else "")


def lead(rows):
    hits = []
    for g, threshold in LEAD.items():
        p, b = rows[g]
        lo, hi = np.percentile(b, [2.5, 97.5])
        if abs(p) >= threshold and (lo > 0 or hi < 0):
            hits.append(f"{g} {'+' if p > 0 else '−'}")
    return ", ".join(hits)


def summary_table(diffs):
    return pd.DataFrame({c: {**{g: interval(*rows[g]) for g in GROUPS if g in rows}, "lead": lead(rows)}
                         for c, rows in diffs.items()}).T


def numbers(diffs):
    return [{"comparison": c, "rows": r, "difference": p, "95% low": np.percentile(b, 2.5),
             "95% high": np.percentile(b, 97.5)} for c, rows in diffs.items() for r, (p, b) in rows.items()]


boot_direct = bootstrap(S_direct, types_direct)
assert np.allclose(100 * task_weights(boot_direct[0], GROUPS["overall"]) @ boot_direct[1],
                   scores.loc[DIRECT_RUN, "overall"].to_numpy(float))
pairs = [(x, y) for x, y in COMPARISONS if x in DIRECT_RUN and y in DIRECT_RUN]
DIFFS = {f"{x} − {y}": difference(boot_direct, x, y) for x, y in pairs}
VS_VIDEO = ({f"{x} − video": difference(boot_direct, x, "video") for x in DIRECT_RUN if x != "video"}
            if "video" in DIRECT_RUN else {})
diff_numbers = numbers({**DIFFS, **VS_VIDEO})
print(f"{len(D_IDS)} questions answered in every direct condition; {B:,} stratified paired resamples")
print("The comparisons (README):")
display(summary_table(DIFFS))
print("Every direct condition against video only:")
display(summary_table(VS_VIDEO))
print("The comparisons per question type:")
per_type_diffs = pd.DataFrame({c: {t: interval(*rows[t]) for t in boot_direct[0]} for c, rows in DIFFS.items()})
per_type_diffs.insert(0, "n", by_type["n"])
display(per_type_diffs)

## Metres against relative units

Each direct sheet condition here against the same condition in run 1, in relative units (`RELATIVE_DIR`), on the
questions both answered: what writing the sheet in metres changes. The prompts differ only in the sheet's numbers and
the words for its units. The **metric tasks** row is the one the metres are for.

In [ ]:
rel_names = [n for n in DIRECT_RUN if n != "video" and (Path(RELATIVE_DIR) / f"{n}.jsonl").exists()]
if not rel_names:
    print(f"no run-1 predictions in {RELATIVE_DIR}: nothing to compare")
    METRES_DIFFS = {}
else:
    rel = {}
    for n in rel_names:
        r = pd.read_json(Path(RELATIVE_DIR) / f"{n}.jsonl", lines=True).drop_duplicates("id").set_index("id")
        shared = [q for q in r.index if q in DOCS]  # this run's questions
        rel[f"{n} (relative)"] = pd.Series([score_final(DOCS[q], str(r.loc[q, "prediction"])) for q in shared], index=shared)
    R_IDS = [q for q in D_IDS if all(q in s.index for s in rel.values())]
    S_both = pd.concat([S_direct.loc[R_IDS, rel_names], pd.DataFrame(rel).loc[R_IDS]], axis=1)
    boot_both = bootstrap(S_both, types_direct.loc[R_IDS])
    METRES_DIFFS = {f"{n}: metres − relative": difference(boot_both, n, f"{n} (relative)") for n in rel_names}
    diff_numbers += numbers(METRES_DIFFS)
    print(f"{len(R_IDS)} questions answered in both runs")
    display(summary_table(METRES_DIFFS))
    print("Per task:")
    display(pd.DataFrame({c: {t: interval(*rows[t]) for t in boot_both[0]} for c, rows in METRES_DIFFS.items()}).T)

## Summary: the CoT conditions

On the questions the CoT conditions ran on (all 200, or `SUBSET`'s), each CoT condition against its direct one (the
same sheet, answering at once), then the three against each other, and every condition against `video`. The README's
reading rule applies. The CoT pilot found that structured CoT does not help Qwen3.5-2B with a gold sheet overall
(−3.3, interval −10.5 to +4.0) and hurts without a sheet (−13.3), on these questions.

In [ ]:
COT_NAMES = [n for n in COT_RUN if n in runs]
if not COT_NAMES:
    print("no CoT predictions yet")
else:
    shown = ["video"] + [COT_BASES[n] for n in COT_NAMES] + COT_NAMES
    C_IDS = [q for q in RUN_IDS if all(q in runs[n].index for n in shown)]
    S_cot = scores_of(shown, C_IDS)
    types_cot = pd.Series({q: TYPE[q] for q in C_IDS})
    cot_scores = score_table(S_cot, types_cot, shown)
    print(f"{len(C_IDS)} questions; per task: {dict(Counter(types_cot.map(task_of)))}")
    display(cot_scores.round(1))
    boot_cot = bootstrap(S_cot, types_cot)
    COT_DIFFS = {f"{x} − {y}": difference(boot_cot, x, y) for x, y in cot_comparisons(COT_RUN)
                 if x in shown and y in shown}
    COT_DIFFS.update({f"{x} − video": difference(boot_cot, x, "video") for x in COT_NAMES})
    diff_numbers += numbers(COT_DIFFS)
    display(summary_table(COT_DIFFS))
    print("Per question type:")
    display(pd.DataFrame({c: {t: interval(*rows[t]) for t in boot_cot[0]} for c, rows in COT_DIFFS.items()}).T)

## The answer letters

Run 1 found that the sheets shift the model's multiple-choice answers to B, and that a condition can gain on a task
from that alone. For every condition on the multiple-choice questions it ran on: the share of B answers, and how many
are right when the true answer is B and when it is not. Then the score of always giving one letter.

In [ ]:
def letter_row(name, qs):
    lt = runs[name].loc[qs, "read"]
    t = pd.Series({q: DOCS[q]["ground_truth"] for q in qs})
    not_b = t != "B"
    return {"questions": len(qs), "truth B": int((~not_b).sum()), "answers B %": 100 * (lt == "B").mean(),
            "right, truth B %": 100 * (lt[~not_b] == "B").mean(), "right, truth not B %": 100 * (lt[not_b] == t[not_b]).mean(),
            "right, all %": 100 * (lt == t).mean()}


DIR_ROUTE = GROUPS["direction and route"]
for label, keep in (("all multiple-choice questions", lambda q: TYPE[q] in MCA_QUESTION_TYPES),
                    ("direction and route", lambda q: task_of(TYPE[q]) in DIR_ROUTE)):
    letters = pd.DataFrame({n: letter_row(n, [q for q in r.index if keep(q)]) for n, r in runs.items()}).T
    print(f"{label}:")
    display(letters.round(1))
mca = [q for q in IDS if TYPE[q] in MCA_QUESTION_TYPES]
const = {}
for L in "ABCD":
    df = pd.DataFrame({"question_type": [TYPE[q] for q in mca], "score": [float(DOCS[q]["ground_truth"] == L) for q in mca]})
    s = summarize(df)
    const[f"always {L}"] = {k: s[k] for k in TASKS if k in s}
print("always one letter, all 200 questions' multiple-choice tasks:")
display(pd.DataFrame(const).T.round(1))

## The numeric answers

Median answer per numeric question type on all 200 questions, with the true median: with the sheets in metres, the
distances and sizes the model gives should no longer be the sheet's units read as metres (run 1: absolute distance
about 1.0 against 1.9 m true).

In [ ]:
numeric = {}
for qt in ["object_counting", "object_abs_distance", "object_size_estimation", "room_size_estimation"]:
    qs = [q for q in IDS if TYPE[q] == qt]
    truth = np.median([float(DOCS[q]["ground_truth"]) for q in qs])
    numeric[f"{qt} (truth {truth:g})"] = {n: np.nanmedian([np.nan if pd.isna(r.loc[q, "read"]) else float(r.loc[q, "read"])
                                                           for q in qs if q in r.index] or [np.nan]) for n, r in runs.items()}
display(pd.DataFrame(numeric).round(2))

## Reasoning check: the CoT replies

Over each CoT condition's replies whose sheet has objects: a reply **names a row** if it contains one of the sheet's
object ids (such as `chair_2`), **copies an object number** if it contains one of the objects' positions or extents as
the sheet writes them (in metres), and **copies a camera number** if it contains a number the sheet gives only for
the camera: a camera table's positions, the compass path's move lengths, or the narrative's move lengths. These are
the structured-CoT pilot's checks (`make_table.py` there), with the camera added. **Right**: the exact option for
multiple choice, a score of at least 0.5 for numbers.

In [ ]:
DECIMAL = re.compile(r"-?\d+\.(\d{2,})")
MOVE = re.compile(r"\b(?:[Mm]ove|Go) (\d+\.\d{3}) metres")


def copies_number(reply, values):
    """The pilot's rule: a number in the reply with 2 or more decimals that equals a sheet value at that precision."""
    for m in DECIMAL.finditer(reply):
        d = min(len(m.group(1)), ss.DECIMALS)
        x = float(m.group(0))
        if any(round(v, d) == round(x, d) for v in values):
            return True
    return False


def names_row(reply, ids):
    text = reply.lower()
    return any(re.search(rf"(?<![a-z0-9_]){re.escape(i)}(?![a-z0-9_])", text) for i in ids)


def sheet_facts(q, name):
    """The sheet's object ids, its objects' numbers, and the numbers it gives only for the camera."""
    o = opts(name)
    scale = ss.scale_of(q, ROOT, METRIC_SCALE)
    objects = ss.load_objects(q, "pipeline", ROOT, vggt_root=VGGT_DIR, scale=scale)
    values = {float(obj[k]) for obj in objects for k in ss.POSITION + ss.EXTENT}
    if o["fmt"] == "narrative" or o["fmt"] == "text":
        camera = {float(x) for x in MOVE.findall(SHEET[name][q])}
    else:
        camera = {float(c[k]) for c in ss.load_cameras(q, ROOT, VGGT_DIR, scale=scale) for k in ss.POSITION}
    return {obj["id"].lower() for obj in objects}, values, camera - values


COMPUTING = {"object_abs_distance", "object_rel_distance", "object_size_estimation"}
check = []
for name in COT_NAMES:
    for q, r in runs[name].iterrows():
        if ss.EMPTY in SHEET[name][q]:
            continue
        ids_, values, camera = sheet_facts(q, name)
        right = r["rescored"] == 1 if TYPE[q] in MCA_QUESTION_TYPES else r["rescored"] >= 0.5
        check.append({"condition": name, "task": task_of(TYPE[q]), "cut": r["hit_token_limit"],
                      "names a row": names_row(r["prediction"], ids_), "copies an object number": copies_number(r["prediction"], values),
                      "copies a camera number": copies_number(r["prediction"], camera), "right": right})
check = pd.DataFrame(check)
if len(check):
    rows = []
    for name, g in check.groupby("condition", sort=False):
        for label, m in (("all", g.index), ("computing tasks", g.index[g["task"].isin(COMPUTING)]),
                         ("direction and route", g.index[g["task"].isin(DIR_ROUTE)]), ("cut off", g.index[g["cut"]])):
            sub = g.loc[m]
            rows.append({"condition": name, "replies": label, "n": len(sub),
                         **{c: 100 * sub[c].mean() if len(sub) else np.nan
                            for c in ("names a row", "copies an object number", "copies a camera number", "right")}})
    reasoning_check = pd.DataFrame(rows).set_index(["condition", "replies"])
    display(reasoning_check.round(0))
else:
    reasoning_check = pd.DataFrame()

## Example replies

For each family's main direct comparison, one question the first condition gets right and the second wrong, and one the
other way round, drawn at random (`SEED`), from relative direction and route planning when there are any. Then, for
each CoT condition, one right reply, one wrong one and one cut off at the token limit, with the sheet it was given
(shortened), the right and wrong ones from relative direction and route planning when there are any.

In [ ]:
SEED = 0
EXAMPLE_PAIRS = [("cam_csv", "obj_csv"), ("cam_markdown_table", "obj_markdown_table"), ("cam_text", "obj_text"),
                 ("narrative", "cam_text")]
CAMERA_TYPES = {t for t in TYPE_ORDER if task_of(t) in DIR_ROUTE}
rng_examples = random.Random(SEED)


def shorten(text, limit=4000):
    if len(text) <= limit:
        return text
    return text[:2 * limit // 3] + f"\n\n[... {len(text) - limit} characters left out ...]\n\n" + text[-limit // 3:]


def show_question(q):
    d = DOCS[q]
    print(f"question {q} ({TYPE[q]}, {d['dataset']}/{d['scene_name']}): {d['question']}")
    if TYPE[q] in MCA_QUESTION_TYPES:
        print("\n".join(d["options"]))
    print(f"ground truth: {d['ground_truth']}")


for x, y in EXAMPLE_PAIRS:
    if x not in S_direct or y not in S_direct:
        continue
    for better, worse in ((x, y), (y, x)):
        gap = S_direct[better] - S_direct[worse]
        pool = sorted(q for q in D_IDS if gap[q] >= 0.5)
        pool = [q for q in pool if TYPE[q] in CAMERA_TYPES] or pool
        print("=" * 100)
        if not pool:
            print(f"{better} better than {worse}: no such question")
            continue
        q = rng_examples.choice(pool)
        print(f"{better.upper()} BETTER THAN {worse.upper()} ({len(pool)} such questions):")
        show_question(q)
        for name in (x, y):
            r = runs[name].loc[q]
            print(f"  {name}: reply {shorten(r['prediction'], 600)!r}, read {r['read']!r}, score {r['rescored']:.2f}")
        print("-" * 30, f"{x}'s scene information")
        print(shorten(SHEET[x][q]))

for name in COT_NAMES:
    r_all = runs[name]
    cut = r_all["hit_token_limit"]
    for kind in ("right", "wrong", "cut"):
        if kind == "right":
            pool = list(r_all.index[(r_all["rescored"] == 1) & ~cut])
        elif kind == "wrong":
            pool = list(r_all.index[(r_all["rescored"] == 0) & r_all["parsed"] & ~cut])
        else:
            pool = list(r_all.index[cut & r_all["line"].isna()]) or list(r_all.index[cut])
        if kind != "cut":
            pool = [q for q in pool if TYPE[q] in CAMERA_TYPES] or pool
        print("=" * 100)
        if not pool:
            print(f"{name}, {kind}: no such reply")
            continue
        q = rng_examples.choice(sorted(pool))
        r = r_all.loc[q]
        print(f"{kind.upper()}: {name}")
        show_question(q)
        print(f"the rule reads {r['read']!r}, score {r['rescored']:.2f}; {r['n_new_tokens']} tokens"
              + (f", cut off at {COT_TOKENS}" if r["hit_token_limit"] else ""))
        print("-" * 30, "reply")
        print(shorten(r["prediction"], 6000))
        print("-" * 30, "the sheet it was given")
        print(shorten(SHEET[name][q], 3000))

## Save

In [ ]:
# The tables next to the predictions (on Drive when USE_DRIVE), and what this run was.
table.to_csv(f"{RESULTS_DIR}/per_question.csv")
answers.to_csv(f"{RESULTS_DIR}/answers.csv")
scores.to_csv(f"{RESULTS_DIR}/scores.csv")
by_type.to_csv(f"{RESULTS_DIR}/by_question_type.csv")
lengths.to_csv(f"{RESULTS_DIR}/input_tokens.csv")
if COT_NAMES:
    cot_scores.to_csv(f"{RESULTS_DIR}/scores_cot.csv")
    per_type_cot.to_csv(f"{RESULTS_DIR}/cot_replies_by_question_type.csv")
    reasoning_check.to_csv(f"{RESULTS_DIR}/reasoning_check.csv")
pd.DataFrame(diff_numbers).to_csv(f"{RESULTS_DIR}/differences_bootstrap.csv", index=False)
Path(f"{RESULTS_DIR}/run_info.json").write_text(json.dumps({
    "code_zip": CODE_ZIP, "code_commit": CODE_COMMIT, "gpu": GPU, "model": MODEL, "common": COMMON,
    "metric_scale": METRIC_SCALE, "conditions": {n: opts(n) for n in ACTIVE},
    "eval_args": {n: eval_args(n) for n in ACTIVE}, "cot_table": json.loads(COT_TABLE_JSON.read_text()),
    "cot_subset": globals().get("CHOICE"), "direct_questions": len(D_IDS),
    "cot_questions": len(globals().get("C_IDS", [])), "reused": reused, "timing": timing}, indent=1))
print("saved to", RESULTS_DIR)